In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data
import torchvision
import matplotlib.pyplot as plt
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, random_split
from torch.utils.data import random_split, Subset
from sklearn.model_selection import train_test_split
from PIL import Image
from tqdm.notebook import tqdm
from torch.utils.tensorboard import SummaryWriter
import numpy as np
import torch
import torchinfo
from torchvision.datasets import CIFAR10, CIFAR100
from datetime import datetime
from pathlib import Path
import random
import json
import torch
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# 1. Setup Base Paths
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

data_dir = Path("data")
base_dir = Path("transfer")
run_dir  = base_dir / f"run_{timestamp}"

models = ["dae", "clf_trasfer", "clf_scratch"]
dirs = {}

# 2. Define Hierarchy
for model in models:
    model_dir = run_dir / model
    dirs[model] = {
        "root": model_dir,
        "logs": model_dir / "logs",
        "checkpoints": model_dir / "checkpoints",
        "configs": model_dir / "configs",
        "results": model_dir / "results",
    }

# 3. Create folders
data_dir.mkdir(parents=True, exist_ok=True)

for model_paths in dirs.values():
    for path in model_paths.values():
        path.mkdir(parents=True, exist_ok=True)

# Example Usage:
print(f"AE Logs will be saved to: {dirs['dae']['logs']}")


In [ ]:
def seed_everything(seed=42):
    # 1. Basic Python and OS Level
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    
    # 2. Torch CPU
    torch.manual_seed(seed)
    
    # 3. NVIDIA CUDA (Nvidia GPUs)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # for multi-GPU
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        
    # 4. Apple MPS (M1/M2/M3 Macs)
    if hasattr(torch, "mps") and torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)
        # Note: torch.use_deterministic_algorithms(True) often fails on MPS
        # because Metal shaders don't support it for all ops yet.
    
    # 5. Global Determinism (Optional/Strict)
    # This will throw an error if an op doesn't have a deterministic impl.
    # torch.use_deterministic_algorithms(True, warn_only=True)


In [ ]:
seed_everything(42)

In [ ]:
def get_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    elif torch.cuda.is_available():
        return torch.device("cuda")
    else:
        return torch.device("cpu")
    

In [ ]:
device = get_device()
print(device)

In [ ]:
# Transformations applied on each image => only make them a tensor
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

# Loading the training dataset. We need to split it into a training and validation part
cifar100_train_dataset = CIFAR100(root=data_dir, train=True,  transform=transform, download=True)
# Loading the test set
cifar100_test_set      = CIFAR100(root=data_dir, train=False, transform=transform, download=True)

cifar100_train_set, cifar100_val_set = torch.utils.data.random_split(cifar100_train_dataset, [45000, 5000])

batch_size = 256
pin_memory = get_device() == "cuda"
num_workers = 0
# We define a set of data loaders that we can use for various purposes later.

cifar100_train_loader = data.DataLoader(cifar100_train_set, batch_size=batch_size, 
                                       shuffle=True,  drop_last=True, 
                                       pin_memory=pin_memory, num_workers=num_workers)

cifar100_val_loader   = data.DataLoader(cifar100_val_set, batch_size=batch_size, 
                                       shuffle=False, drop_last=False, 
                                       pin_memory=pin_memory, num_workers=num_workers)

cifar100_test_loader  = data.DataLoader(cifar100_test_set, batch_size=batch_size, 
                                       shuffle=False, drop_last=False, 
                                       pin_memory=pin_memory, num_workers=num_workers)


In [ ]:
class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = np.inf
        self.best_acc = -np.inf
        self.early_stop = False
        self.best_state = None
        self.best_epoch = -1

    def __call__(self, model, epoch, val_loss, val_acc=None):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.best_epoch = epoch
            self.counter = 0
            if val_acc:
                self.best_acc = val_acc

            self.best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
        else:
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop

    def load_best_weights(self, model):
        if self.best_state is not None:
            model.load_state_dict(self.best_state)
            print(f"Restored best model weights (Val Loss: {self.best_loss:.4f})")


In [ ]:
class AE_Encoder(nn.Module):
    def __init__(self, num_input_channels=3, base_channel_size=64, latent_dim=256):
        super().__init__()
        self.latent_dim = latent_dim
        # 32x32 → 16x16
        self.block1 = nn.Sequential(
            nn.Conv2d(num_input_channels, base_channel_size, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size),
            nn.GELU(),
            nn.Conv2d(base_channel_size, base_channel_size, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size),
            nn.GELU(),
        )

        # 16x16 → 8x8
        self.block2 = nn.Sequential(
            nn.Conv2d(base_channel_size, base_channel_size * 2, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size * 2),
            nn.GELU(),
            nn.Conv2d(base_channel_size * 2, base_channel_size * 2, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size * 2),
            nn.GELU(),
        )

        # 8x8 → 4x4
        self.block3 = nn.Sequential(
            nn.Conv2d(base_channel_size * 2, base_channel_size * 4, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size * 4),
            nn.GELU(),
            nn.Conv2d(base_channel_size * 4, base_channel_size * 4, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size * 4),
            nn.GELU(),
        )
        self.pool = nn.AdaptiveAvgPool2d((4, 4))

        # flatten size is now ALWAYS:
        # base_channel_size * 4 * 4 * 4
        self.fc = nn.Linear(base_channel_size * 4 * 4 * 4, latent_dim)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        # ensure fixed spatial size
        x = self.pool(x)
        x = torch.flatten(x, 1)
        z = self.fc(x)
        return z


class AE_Decoder(nn.Module):
    def __init__(self, num_input_channels=3, base_channel_size=64, latent_dim=256):
        super().__init__()
        self.latent_dim = latent_dim
        self.fc = nn.Linear(latent_dim, base_channel_size * 4 * 4 * 4)

        # 4x4 -> 8x8
        self.block1 = nn.Sequential(
            nn.Conv2d(base_channel_size * 4, base_channel_size * 4, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size * 4),
            nn.GELU(),
            # Reduce channels here
            nn.Conv2d(base_channel_size * 4, base_channel_size * 2, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size * 2),
            nn.GELU(),
        )

        # 8x8 -> 16x16
        self.block2 = nn.Sequential(
            nn.Conv2d(base_channel_size * 2, base_channel_size * 2, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size * 2),
            nn.GELU(),
            nn.Conv2d(base_channel_size * 2, base_channel_size, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size),
            nn.GELU(),
        )

        # 16x16 -> 32x32
        self.block3 = nn.Sequential(
            nn.Conv2d(base_channel_size, base_channel_size, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size),
            nn.GELU(),
            nn.Conv2d(base_channel_size, num_input_channels, kernel_size=3, padding=1, bias=True),
        )

        self.output_act = nn.Tanh()

    def forward(self, z):
        x = self.fc(z)
        x = x.view(z.size(0), -1, 4, 4)
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        x = self.block1(x)
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        x = self.block2(x)
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        x = self.block3(x)
        return self.output_act(x)

class AE(nn.Module):
    def __init__(self, num_input_channels=3, base_channel_size=64, latent_dim=256):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = AE_Encoder(
            num_input_channels=num_input_channels,
            base_channel_size=base_channel_size,
            latent_dim=latent_dim
        )
        self.decoder = AE_Decoder(
            num_input_channels=num_input_channels,
            base_channel_size=base_channel_size,
            latent_dim=latent_dim
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat


In [ ]:
def reconstruction_loss(y_pred, y_true):
    loss = F.mse_loss(y_pred, y_true, reduction='none')
    loss = loss.view(loss.size(0), -1).sum(dim=1)
    return loss.mean(dim=0)


In [ ]:
def get_AdamW_CosineAnnealingLR(model, lr, weight_decay, T_max, eta_min):
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=T_max, eta_min=eta_min)
    return optimizer, scheduler


In [ ]:
class GaussianNoise:
    def __init__(self, mean=0.0, std=0.2, clamp_min=-1.0, clamp_max=1.0):
        self.mean = mean
        self.std = std
        self.clamp_min = clamp_min
        self.clamp_max = clamp_max

    def __call__(self, images):
        """
        images: Tensor [B, C, H, W] or [C, H, W] in [-1, 1]
        """
        if not self.std or self.std == 0:
            return images

        noise = torch.randn_like(images) * self.std + self.mean
        noisy = images + noise

        return torch.clamp(noisy, self.clamp_min, self.clamp_max)


In [ ]:
def train_autoencoder(train_loader, val_loader, test_loader, 
          model, criterion, noise,
          optimizer, scheduler, 
          max_epochs, early_stopping,
          writer, checkpoint_filename, result_filename): 
    
    device = next(model.parameters()).device
    print(f"Training on {device}")
    
    for epoch in range(max_epochs):
        # -------- TRAIN --------
        model.train()
        train_loss_sum = 0.0
        train_num_samples = 0
        train_bar = tqdm(train_loader, desc=f"Epoch {epoch:03d} [Train]")
        for x, _ in train_bar:
            batch_size = x.size(0)
            train_num_samples += batch_size
            x = x.to(device)
            x_new = noise(x) if noise else x
            optimizer.zero_grad(set_to_none=True)
            x_hat = model(x_new)
            loss = criterion(x_hat, x)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()

            train_loss_sum += loss.item() * batch_size            
            train_bar.set_postfix(loss=loss.item())

        train_loss = train_loss_sum / train_num_samples

        # -------- VALIDATION --------
        model.eval()
        val_loss_sum = 0.0
        val_num_samples = 0
        
        val_bar = tqdm(val_loader, desc=f"Epoch {epoch:03d} [Val]")
        with torch.no_grad():
            for x, _ in val_bar:
                batch_size = x.size(0)
                val_num_samples += batch_size
                x = x.to(device)
                x_new = noise(x) if noise else x
                x_hat = model(x_new)
                loss = criterion(x_hat, x)
                val_loss_sum += loss.item() * batch_size
                val_bar.set_postfix(loss=loss.item())
        
        val_loss = val_loss_sum / val_num_samples

        # -------------------- SCHEDULER --------------------
        if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
            scheduler.step(val_loss)
        else:
            scheduler.step()

        # -------------------- LOGGING --------------------
        writer.add_scalar("loss/train", train_loss, epoch)
        writer.add_scalar("loss/val",   val_loss, epoch)
        writer.add_scalar("lr", optimizer.param_groups[0]["lr"], epoch)

        print(
            f"Epoch {epoch:03d} | "
            f"Train: Loss = {train_loss:.4f} | Val: Loss = {val_loss:.4f}"
        )
        
        # -------------------- EARLY STOPPING --------------------

        if early_stopping(model, epoch, val_loss):
            print("Stopping early")
            break
            
    # Before saving or testing, revert to the best discovered state
    early_stopping.load_best_weights(model)
    ## -------------------- SAVE --------------------
    #torch.save(
    #    model.state_dict(),
    #    os.path.join(checkpoint_path, f"{model.name}_{model.latent_dim}.pt")
    #)
    torch.save({
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "best_epoch": early_stopping.best_epoch,
    }, checkpoint_filename)
    # -------------------- TEST --------------------
    model.eval()
    test_loss_sum = 0.0
    test_num_samples = 0
    with torch.no_grad():
        for x, _ in test_loader:
            batch_size = x.size(0)
            test_num_samples += batch_size
            x = x.to(device)
            x_new = noise(x) if noise else x
            x_hat = model(x_new)
            loss = criterion(x_hat, x)
            test_loss_sum += loss.item() * batch_size

    test_loss = test_loss_sum / test_num_samples
    print(f"Test: Loss = {test_loss:.4f}")  
    # After the test results are calculated
    
    result = {
        "test": {"loss": test_loss, },
        "val":  {"loss": early_stopping.best_loss, 
                 "best_epoch": early_stopping.best_epoch,
                },
    }
    # Save results next to the checkpoint
    with open(result_filename, "w") as f:
        json.dump(result, f, indent=4)
    
    print(f"Results saved to {result_filename}")
    return result


In [ ]:
#def get_train_images(dataset, num):
#    return torch.stack([dataset[i][0] for i in range(num)], dim=0)

def visualize_denoising(dataset, num_samples=6, model=None):
    """
    Takes samples from a dataset, adds noise, runs them through the model,
    and plots a 3-row comparison: Original, Noisy, and Reconstructed.
    """
    
    # 1. Get sample images
    # Assuming dataset[i] returns (image, label)
    indices = torch.randperm(len(dataset))[:num_samples]
    original_images = torch.stack([dataset[i][0] for i in indices])
    
    # 2. Apply Noise
    noise_func = GaussianNoise() # Ensure this class is defined in your scope
    noisy_images = noise_func(original_images)
    comparison_list = [original_images, noisy_images]
    # 3. Model Inference (Reconstruction)
    if model:
        model.eval()
        device = next(model.parameters()).device
        with torch.no_grad():
            reconstructed = model(noisy_images.to(device))
        comparison_list.append(reconstructed.cpu())

    # 4. Stack for the Grid: (3 rows, num_samples columns)
    # Stack along dim=0 creates (3, num_samples, C, H, W)
    # flatten(0, 1) makes it (3 * num_samples, C, H, W) for make_grid
    comparison_stack = torch.stack(comparison_list, dim=0).flatten(0, 1)
    
    # 5. Create Grid
    grid = torchvision.utils.make_grid(
        comparison_stack.cpu(), 
        nrow=num_samples, # Sets the number of columns
        normalize=True, 
        value_range=(-1, 1), 
        padding=2
    )
    
    # 6. Plotting
    plt.figure(figsize=(num_samples * 2, 2 * len(comparison_list)))
    plt.imshow(grid.permute(1, 2, 0))
    title = "Rows: Original (Top), Noisy " + "(Bottom)" if model is None else "(Middle), Reconstructed (Bottom)"
    plt.title(title)
    plt.axis("off")
    plt.show()

# Example Usage:
visualize_denoising(cifar100_test_set, 6)

In [ ]:
MAX_EPOCHS_DAE = 85
dae_hparams = {
    "model": {"latent_dim": 256, "num_input_channels": 3, "base_channel_size":64},
    "training": {"max_epochs": MAX_EPOCHS_DAE, "device": str(device),}, 
    "opt": {"lr": 1e-3, "weight_decay": 1e-4, "T_max": MAX_EPOCHS_DAE, "eta_min": 1e-6, },
    "early_stopping":{"patience": 15, "min_delta": 0.001},
    "dataloader": {"batch_size": batch_size, "pin_memory": pin_memory, "num_workers": num_workers},
    "gaussian_noise": {"mean": 0.0, "std": 0.2, "clamp_min": -1.0, "clamp_max": 1.0}
}

with open(dirs['dae']['configs'] / "hyperparams.json", "w") as f:
    json.dump(dae_hparams, f)


In [ ]:
dae_model = AE(**dae_hparams["model"])
print(torchinfo.summary(dae_model, input_size=(batch_size, 3, 32, 32)))
dae_model.to(device)

dae_criterion = reconstruction_loss
dae_optimizer, dae_scheduler = get_AdamW_CosineAnnealingLR(dae_model, **dae_hparams["opt"])
dae_early_stopping = EarlyStopping(**dae_hparams["early_stopping"])

dae_writer = SummaryWriter(dirs['dae']['logs'])
dae_checkpoint_filename = dirs['dae']['checkpoints'] / "model.pt"
dae_result_filename = dirs['dae']['results'] / "result.json"
dae_noise = GaussianNoise(mean=0.0, std=0.2, clamp_min=-1.0, clamp_max=1.0)


In [ ]:
dae_result = train_autoencoder(cifar100_train_loader, cifar100_val_loader, cifar100_test_loader,
                  dae_model, dae_criterion, dae_noise,
                  dae_optimizer, dae_scheduler, 
                  MAX_EPOCHS_DAE, dae_early_stopping,
                  dae_writer, dae_checkpoint_filename, dae_result_filename
                 )


In [ ]:
visualize_denoising(cifar100_test_set, 6, dae_model)


In [ ]:
def cross_entropy_loss(logits, target):
    """
    pred: (N, C) logits
    target: (N,) class labels
    """
    logits = logits.view(-1, logits.shape[-1]).to(torch.float32)   # ensure (N, C)
    target = target.view(-1).to(torch.float32)                # ensure (N,)
    return F.cross_entropy(logits, target)


In [ ]:
train_cifar10_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10, fill=(128,128,128)),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.1,
        hue=0.02
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        (0.5,0.5,0.5),
        (0.5,0.5,0.5)
    ),

    transforms.RandomErasing(
        p=0.25,
        scale=(0.02,0.12),
        ratio=(0.3,3.3),
        value=0
    ),
])

test_cifar10_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),
                         (0.5,0.5,0.5))
])


batch_size = 256
pin_memory = get_device() == "cuda"
num_workers = 0

cifar10_train_base_set = CIFAR10(root=data_dir, train=True, download=True)
#indices = torch.randperm(len(cifar10_train_fullset))
#train_idx = indices[:45000]
#val_idx   = indices[45000:]
#labels = base_dataset._labels
labels = [cifar10_train_base_set[i][1] for i in range(len(cifar10_train_base_set))]
indices = np.arange(len(cifar10_train_base_set))
# Use train_test_split to get lists of integers
train_idx, val_idx = train_test_split(indices, train_size=0.9, random_state=42, stratify=np.array(labels))

cifar10_train_tmp = CIFAR10(root=data_dir, train=True, transform=train_cifar10_transform)
cifar10_val_tmp   = CIFAR10(root=data_dir, train=True, transform=test_cifar10_transform)

cifar10_train_set = Subset(cifar10_train_tmp, train_idx)
cifar10_val_set   = Subset(cifar10_val_tmp,   val_idx)

cifar10_test_set  = CIFAR10(root=data_dir, train=False, transform=test_cifar10_transform, download=True)

# Loading the test set

cifar10_train_loader = data.DataLoader(cifar10_train_set, batch_size=batch_size, 
                                       shuffle=True,  drop_last=True, 
                                       pin_memory=pin_memory, num_workers=num_workers)

cifar10_val_loader   = data.DataLoader(cifar10_val_set, batch_size=batch_size, 
                                       shuffle=False, drop_last=False, 
                                       pin_memory=pin_memory, num_workers=num_workers)

cifar10_test_loader  = data.DataLoader(cifar10_test_set, batch_size=batch_size, 
                                       shuffle=False, drop_last=False, 
                                       pin_memory=pin_memory, num_workers=num_workers)




In [ ]:
def get_norm(parameters):
    total_sq = 0.0
    for p in parameters:
        # ONLY count if the parameter is actually being updated
        if p.requires_grad and p.grad is not None:
            total_sq += p.grad.detach().norm(2).item() ** 2
    return total_sq ** 0.5

def grad_norm_stats(model):
    return {
        "encoder": get_norm(model.encoder.parameters()),
        "head": get_norm(model.head.parameters()),
    }


In [ ]:
def train_classifier(train_loader, val_loader, test_loader, 
          model, criterion, 
          optimizer, scheduler, 
          max_epochs, early_stopping,
          writer, checkpoint_filename, result_filename): 
    grad_norm_stat = {}
    device = next(model.parameters()).device
    print(f"Training on {device}")
    is_stage_one = (any(p.requires_grad for p in model.encoder.parameters()) == False)
    stage = 1 if is_stage_one else 2
    print(f"This is stage {stage}")
    for epoch in range(max_epochs):
        # -------- TRAIN --------
        model.train()
        if is_stage_one:
            model.encoder.eval() # Only force eval if we are frozen
            model.encoder.zero_grad(set_to_none=True)
        train_loss_sum = 0.0
        train_correct = 0
        train_num_samples = 0
        train_bar = tqdm(train_loader, desc=f"Epoch {epoch:03d} [Train]")
        for x, y in train_bar:
            batch_size = x.size(0)
            train_num_samples += batch_size
            x = x.to(device)
            y = y.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            grad_sum = grad_norm_stats(model)
            optimizer.step()
            # stats
            train_loss_sum += loss.item() * x.size(0)
            preds = logits.argmax(dim=1)
            train_correct += (preds == y).sum().item()       
            train_bar.set_postfix(loss=loss.item())
        
        print(f"Encoder grad norm sum: encoder={grad_sum['encoder']:.6f} head={grad_sum['head']:.6f}")
        train_loss = train_loss_sum / train_num_samples
        train_acc = train_correct / train_num_samples
        if epoch % 5 == 0:
            for name, param in model.named_parameters():
                if param.grad is not None:
                    print(f"{name:30} | Norm: {param.grad.norm().item():.4f}")
                    grad_norm_stat[f"{name}"] = param.grad.norm().item()
                    

        # -------- VALIDATION --------
        model.eval()
        val_loss_sum = 0.0
        val_correct = 0
        val_num_samples = 0
        
        val_bar = tqdm(val_loader, desc=f"Epoch {epoch:03d} [Val]")
        with torch.no_grad():
            for x, y in val_bar:
                batch_size = x.size(0)
                val_num_samples += batch_size
                x = x.to(device)
                y = y.to(device)
                logits = model(x)
                loss = criterion(logits, y)
                val_loss_sum += loss.item() * batch_size
                preds = logits.argmax(dim=1)
                val_correct += (preds == y).sum().item()

                val_bar.set_postfix(loss=loss.item())

        val_loss = val_loss_sum / val_num_samples
        val_acc  = val_correct / val_num_samples
        
        # -------------------- SCHEDULER --------------------
        if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
            scheduler.step(val_loss)
        else:
            scheduler.step()

        # -------------------- LOGGING --------------------
        writer.add_scalar("loss/train", train_loss, epoch)
        writer.add_scalar("loss/val", val_loss, epoch)

        writer.add_scalar("acc/train", train_acc, epoch)
        writer.add_scalar("acc/val", val_acc, epoch)
        
        writer.add_scalar("grad/encoder_norm", grad_sum["encoder"], epoch)
        writer.add_scalar("grad/head_norm", grad_sum["head"], epoch)
        
        writer.add_scalar("lr", optimizer.param_groups[0]["lr"], epoch)
        if epoch % 5 == 0:
            for name, grad in grad_norm_stat.items():
                writer.add_scalar(name, grad, epoch)
    
        print(
            f"Epoch {epoch:03d} | "
            f"Train: Loss = {train_loss:.4f} Acc = {train_acc:.4f} | "
            f"Val: Loss = {val_loss:.4f} Acc = {val_acc:.4f}"
        )

        
        # -------------------- EARLY STOPPING --------------------

        if early_stopping(model, epoch, val_loss, val_acc):
            print("Stopping early")
            break
            
    # Before saving or testing, revert to the best discovered state
    early_stopping.load_best_weights(model)
    ## -------------------- SAVE --------------------
    #torch.save(
    #    model.state_dict(),
    #    os.path.join(checkpoint_path, f"{model.name}_{model.latent_dim}.pt")
    #)
    torch.save({
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "best_epoch": early_stopping.best_epoch,
    }, checkpoint_filename)
    # -------------------- TEST --------------------
    model.eval()
    test_loss_sum = 0.0
    test_correct = 0
    test_num_samples = 0
    with torch.no_grad():
        for x, y in test_loader:
            batch_size = x.size(0)
            test_num_samples += batch_size
            x = x.to(device)
            y = y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            test_loss_sum += loss.item() * batch_size
            preds = logits.argmax(dim=1)
            test_correct += (preds == y).sum().item()

    test_loss = test_loss_sum / test_num_samples
    test_acc  = test_correct / test_num_samples

    print(f"Test: Loss = {test_loss:.4f} Acc = {test_acc:.4f}")

    # After the test results are calculated
    
    result = {
        "test": {"loss": test_loss, "acc": test_acc},
        "val":  {"loss": early_stopping.best_loss,
                 "acc": early_stopping.best_acc,
                 "best_epoch": early_stopping.best_epoch,
                },
    }
    # Save results next to the checkpoint
    with open(result_filename, "w") as f:
        json.dump(result, f, indent=4)
    
    print(f"Results saved to {result_filename}")
    return result


In [ ]:
class ImageClassifier(nn.Module):
    def __init__(self, encoder, num_classes=10, dropout_rate=0.3):
        super().__init__()
        # 1. Take the pre-trained encoder
        self.encoder = encoder
        
        # 2. Add a Classification Head
        # We use the latent_dim output from your AE_Encoder's self.fc
        self.head = nn.Sequential(
            nn.Linear(self.encoder.latent_dim, 512, bias=False),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(512, 512, bias=False),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        # Your AE_Encoder returns (z, None)
        z = self.encoder(x)
        
        # Pass the latent representation through the new head
        logits = self.head(z)
        return logits




In [ ]:
def start_stage_one(model, lr, weight_decay, step_size, gamma ):
    # 1. Freeze the parameters
    model.encoder.requires_grad_(False)
    # 2. Set encoder to evaluation mode
    # This freezes BatchNorm statistics and turns off Dropout
    model.encoder.eval()
    # 3. Ensure the head is in training mode
    model.head.train()
    model.head.requires_grad_(True)
    # Verify freeze for peace of mind
    for p in model.encoder.parameters():
        assert p.requires_grad == False
        
    for p in model.head.parameters():
        assert p.requires_grad == True

    optimizer = optim.AdamW(model.head.parameters(), lr=lr, weight_decay=weight_decay)

    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=step_size, gamma=gamma)

    return optimizer, scheduler



In [ ]:
MAX_EPOCHS_CLF1 = 30
MAX_EPOCHS_CLF2 = 85

clf_transfer_hparams = {
    "model": {"num_classes":10, "dropout_rate":0.3},
    "dataloader": {"batch_size": batch_size, "pin_memory": pin_memory, "num_workers": num_workers},
    "stage_1": {
        "training": {"max_epochs": MAX_EPOCHS_CLF1, "device": str(device),}, 
        "opt": {"lr": 1e-3, "weight_decay": 1e-4, "step_size": 10, "gamma":0.5},
        "early_stopping":{"patience": 10, "min_delta": 0.001},
    },
    "stage_2": {
        "training": {"max_epochs": MAX_EPOCHS_CLF2, "device": str(device),}, 
        "opt": {"lr_encoder": 3e-4, "weight_decay_encoder": 1e-4, "lr_head": 1e-3, "weight_decay_head": 1e-4,
            "T_max": MAX_EPOCHS_CLF2, "eta_min":1e-6},
        "early_stopping":{"patience": 10, "min_delta": 0.001},
    }   
}



with open(dirs['clf_trasfer']['configs'] / "hyperparams.json", "w") as f:
    json.dump(clf_transfer_hparams, f)


In [ ]:
with open(dirs['dae']['configs'] / "hyperparams.json", "r") as f:
    dae_config = json.load(f)
    
dae_transfer_model = AE(**dae_config["model"])
clf_transfer_model = ImageClassifier(dae_transfer_model.encoder, **clf_transfer_hparams["model"])

dae_transfer_checkpoint = torch.load(dirs['dae']['checkpoints'] / "model.pt", map_location=device)

dae_transfer_dict = dae_transfer_checkpoint["model_state"]


encoder_dict = {
    k.replace("encoder.", "", 1): v
    for k, v in dae_transfer_dict.items()
    if k.startswith("encoder.")
}

clf_transfer_model.encoder.load_state_dict(encoder_dict, strict=True)
print(torchinfo.summary(clf_transfer_model, input_size=(batch_size, 3, 32, 32)))
clf_transfer_model.to(device)

In [ ]:
criterion = cross_entropy_loss
clf_transfer_optimizer_1, clf_transfer_scheduler_1 = start_stage_one(clf_transfer_model, **clf_transfer_hparams["stage_1"]["opt"])
clf_transfer_early_stopping_1 = EarlyStopping(**clf_transfer_hparams["stage_1"]["early_stopping"])

clf_transfer_writer_1 = SummaryWriter(dirs['clf_trasfer']['logs'] / "stage_1")
clf_transfer_checkpoint_filename_1 = dirs['clf_trasfer']['checkpoints'] / "model_1.pt"
clf_transfer_result_filename_1 = dirs['clf_trasfer']['results'] / "result_1.json"


In [ ]:
clf_transfer_result_1 = train_classifier(cifar10_train_loader, cifar10_val_loader, cifar10_test_loader, 
          clf_transfer_model, criterion, 
          clf_transfer_optimizer_1, clf_transfer_scheduler_1, 
          MAX_EPOCHS_CLF1, clf_transfer_early_stopping_1,
          clf_transfer_writer_1, clf_transfer_checkpoint_filename_1, clf_transfer_result_filename_1)

In [ ]:
@torch.no_grad()
def evaluate_classifier(model, dataloader):
    model.eval()
    device = next(model.parameters()).device
    all_preds = []
    all_labels = []
    
    for x, y in dataloader:
        x, y = x.to(device), y.to(device)
        
        logits = model(x)
        preds = torch.argmax(logits, dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())
    
    # Define CIFAR-10 classes
    classes = ('plane', 'car', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck')
    
    # 1. Classification Report (Precision, Recall, F1)
    print("\n--- Classification Report ---")
    print(classification_report(all_labels, all_preds, target_names=classes))
    
    # 2. Confusion Matrix
    cm = confusion_matrix(all_labels, all_preds)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=classes, yticklabels=classes)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('CIFAR-10 Confusion Matrix')
    plt.show()
    
    return cm


In [ ]:
# After Stage 1
print("Results after Stage 1 (Frozen Encoder):")
cm1 = evaluate_classifier(clf_transfer_model, cifar10_test_loader)

In [ ]:
def start_stage_two(model, lr_encoder, weight_decay_encoder, lr_head, weight_decay_head, T_max, eta_min):
    # Unfreeze all
    for param in model.parameters():
        param.requires_grad = True
        
    # Verify freeze for peace of mind
    for p in model.encoder.parameters():
        assert p.requires_grad == True
        
    for p in model.head.parameters():
        assert p.requires_grad == True    

    optimizer = optim.AdamW([
        {
            "params": model.encoder.parameters(),
            "lr": lr_encoder,
            "weight_decay": weight_decay_encoder
        },
        {
            "params": model.head.parameters(),
            "lr": lr_head,
            "weight_decay": weight_decay_head
        }
    ])

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=T_max,
        eta_min=eta_min
    )
    
    return optimizer, scheduler

In [ ]:
clf_transfer_optimizer_2, clf_transfer_scheduler_2 = start_stage_two(clf_transfer_model, **clf_transfer_hparams["stage_2"]["opt"])
clf_transfer_early_stopping_2 = EarlyStopping(**clf_transfer_hparams["stage_2"]["early_stopping"])

clf_transfer_writer_2 = SummaryWriter(dirs['clf_trasfer']['logs'] / "stage_2")
clf_transfer_checkpoint_filename_2 = dirs['clf_trasfer']['checkpoints'] / "model_2.pt"
clf_transfer_result_filename_2 = dirs['clf_trasfer']['results'] / "result_2.json"


In [ ]:
clf_transfer_result_2 = train_classifier(cifar10_train_loader, cifar10_val_loader, cifar10_test_loader, 
          clf_transfer_model, criterion, 
          clf_transfer_optimizer_2, clf_transfer_scheduler_2, 
          MAX_EPOCHS_CLF2, clf_transfer_early_stopping_2,
          clf_transfer_writer_2, clf_transfer_checkpoint_filename_2, clf_transfer_result_filename_2)

In [ ]:
# After Stage 2
print("Results after Stage 2 (Fine-tuned):")
cm2 = evaluate_classifier(clf_transfer_model, cifar10_test_loader)


In [ ]:
MAX_EPOCHS_SCRATCH = 85

clf_scratch_hparams = {
    "model": {"num_classes":10, "dropout_rate":0.3},
    "dataloader": {"batch_size": batch_size, "pin_memory": pin_memory, "num_workers": num_workers},
    "training": {"max_epochs": MAX_EPOCHS_SCRATCH, "device": str(device),}, 
    "opt": {"lr_encoder": 3e-4, "weight_decay_encoder": 1e-4, "lr_head": 1e-3, "weight_decay_head": 1e-4,
            "T_max": MAX_EPOCHS_SCRATCH, "eta_min":1e-6},
    "early_stopping":{"patience": 10, "min_delta": 0.001},
    }   


with open(dirs['clf_scratch']['configs'] / "hyperparams.json", "w") as f:
    json.dump(clf_scratch_hparams, f)


In [ ]:
dae_scratch_model = AE(**dae_config["model"])

clf_scratch_model = ImageClassifier(dae_scratch_model.encoder, **clf_scratch_hparams["model"])

print(torchinfo.summary(clf_scratch_model, input_size=(batch_size, 3, 32, 32)))
clf_scratch_model.to(device)


In [ ]:
clf_scratch_optimizer, clf_scratch_scheduler = start_stage_two(clf_scratch_model, **clf_scratch_hparams["opt"])
clf_scratch_early_stopping = EarlyStopping(**clf_scratch_hparams["early_stopping"])

clf_scratch_writer = SummaryWriter(dirs['clf_scratch']['logs'])
clf_scratch_checkpoint_filename = dirs['clf_scratch']['checkpoints'] / "model.pt"
clf_scratch_result_filename = dirs['clf_scratch']['results'] / "result.json"


In [ ]:
clf_scratch_result = train_classifier(cifar10_train_loader, cifar10_val_loader, cifar10_test_loader, 
          clf_scratch_model, criterion, 
          clf_scratch_optimizer, clf_scratch_scheduler, 
          MAX_EPOCHS_SCRATCH, clf_scratch_early_stopping,
          clf_scratch_writer, clf_scratch_checkpoint_filename, clf_scratch_result_filename)


In [ ]:
print("Results:")
cm_scratch = evaluate_classifier(clf_scratch_model, cifar10_test_loader)

In [ ]:
# Load and rotate pet
img_pet = Image.open("martino.jpg").convert("RGB")
img_pet = img_pet.transpose(Image.ROTATE_270)
pet_np = np.array(img_pet)


In [ ]:
def preprocess(img, model_img_size=32):
    transform = transforms.Compose([
        transforms.Resize((model_img_size, model_img_size)),
        transforms.ToTensor()
    ])

    img_tensor = transform(img).unsqueeze(0)
    return img_tensor


In [ ]:
def predict(model, input_tensor):
    model.eval()
    device = next(model.parameters()).device
    with torch.no_grad():
        logits = model(input_tensor.to(device))
        preds = torch.argmax(logits, dim=1)
    return logits, preds

In [ ]:
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')


In [ ]:
tensor_pet = preprocess(img_pet, 128)
logits, preds = predict(clf_transfer_model, tensor_pet)
print(f"Martino is a {classes[preds]}")


In [ ]:
tensor_pet = preprocess(img_pet, 128)
logits, preds = predict(clf_scratch_model, tensor_pet)
print(f"Martino is a {classes[preds]}")


# Why Training from Scratch Can Beat Autoencoder Transfer Learning

Training from scratch is not always worse than transfer learning from an autoencoder encoder.

## Main Reason

A denoising autoencoder learns to **reconstruct images**, while a classifier learns to **separate classes**.

These are different objectives.

## Why Scratch Training Can Win

### 1. Better task alignment
Supervised training directly optimizes classification accuracy.

### 2. Compression loss
The autoencoder bottleneck may remove details useful for classification.

### 3. Irrelevant features
Autoencoders may preserve lighting, background, or texture that classifiers should ignore.

### 4. CIFAR has enough labels
Datasets like CIFAR-10 / CIFAR-100 often contain enough labeled data to train good CNNs from scratch.

## When Autoencoder Transfer Helps

- Few labeled examples  
- Large unlabeled dataset available  
- Same-domain pretraining  
- Full fine-tuning after transfer

## Bottom Line

On CIFAR-like datasets, a well-trained CNN from scratch often matches or beats an autoencoder encoder because **classification features are not the same as reconstruction features**.

# Question


What happens if we reduce the CIFAR10 dataset? 

Like considering only the 10% of the training dataset